# MVA WGS + VCF Early Exploration (Kaggle)

This notebook performs **basic technical profiling and exploratory analysis** of VCF and WGS-derived files (FASTQ, BAM, or CRAM). It is designed to stream/chunk large files rather than load a whole genome into RAM.

**Important:** these plots support data QC and candidate discovery; they do not diagnose Mosaic Variegated Aneuploidy (MVA). Confirm findings with validated clinical pipelines and orthogonal testing.


## 1. Kaggle setup
Enable Internet only if package installation requires it. Kaggle input datasets are mounted read-only under `/kaggle/input`; outputs belong in `/kaggle/working`.

In [1]:
import shutil
import subprocess
import sys

missing_hts = [tool for tool in ('bcftools', 'samtools', 'tabix') if shutil.which(tool) is None]
if missing_hts and shutil.which('apt-get'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'bcftools', 'samtools', 'tabix'], check=True, stdout=subprocess.DEVNULL)
    missing_hts = [tool for tool in ('bcftools', 'samtools', 'tabix') if shutil.which(tool) is None]
if missing_hts:
    print('Missing HTS tools:', ', '.join(missing_hts), '| install with Homebrew or apt before VCF/BAM sections.')
try:
    import cyvcf2  # noqa: F401
    import seaborn  # noqa: F401
    import upsetplot  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cyvcf2', 'seaborn', 'upsetplot'], check=True)
print('Setup complete')

Setup complete


In [4]:
pwd

'/Users/macbookpro/Documents/RA/mva-hackathon'

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import os, re, json, gzip, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from cyvcf2 import VCF

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
WORK = Path('/Users/macbookpro/Documents/RA/mva-hackathon') if Path('/Users/macbookpro/Documents/RA/mva-hackathon').exists() else Path.cwd() / 'results'
INPUT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd() / 'data' / 'raw'
OUT = WORK / 'mva_early_insights'
OUT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT)
print('Output:', OUT)

Input: /Users/macbookpro/Documents/RA/mva-hackathon/data/raw
Output: /Users/macbookpro/Documents/RA/mva-hackathon/results/mva_early_insights


## 2. Discover and select files
The code detects VCF/VCF.GZ, BAM/CRAM, and FASTQ files. If several exist, manually replace the selected paths below.

In [3]:
def find_files(root, endings):
    return sorted(p for p in root.rglob('*') if p.is_file() and any(str(p).lower().endswith(e) for e in endings))

vcfs = find_files(INPUT, ('.vcf', '.vcf.gz'))
alignments = find_files(INPUT, ('.bam', '.cram'))
fastqs = find_files(INPUT, ('.fastq', '.fq', '.fastq.gz', '.fq.gz'))
inventory = pd.DataFrame([
    {'type': kind, 'path': str(p), 'size_GB': round(p.stat().st_size / 1e9, 3)}
    for kind, paths in [('VCF', vcfs), ('BAM/CRAM', alignments), ('FASTQ', fastqs)] for p in paths
])
display(inventory if len(inventory) else 'No supported files found under /kaggle/input')

VCF_PATH = vcfs[0] if vcfs else None       # change if needed
ALIGNMENT_PATH = alignments[0] if alignments else None
FASTQ_PATHS = fastqs[:2]                   # typically paired R1/R2
print('VCF:', VCF_PATH)
print('Alignment:', ALIGNMENT_PATH)
print('FASTQ:', FASTQ_PATHS)

'No supported files found under /kaggle/input'

VCF: None
Alignment: None
FASTQ: []


## 3. VCF integrity, header, samples, and index
A compressed VCF is copied to working storage before indexing because `/kaggle/input` is read-only. Re-indexing also resolves old indexes that lack count metadata.

In [ ]:
if VCF_PATH:
    selected_vcf = VCF_PATH
    if str(VCF_PATH).endswith('.vcf.gz'):
        selected_vcf = WORK / VCF_PATH.name
        if not selected_vcf.exists():
            subprocess.run(['cp', str(VCF_PATH), str(selected_vcf)], check=True)
        subprocess.run(['bcftools', 'index', '-f', '-t', str(selected_vcf)], check=True)
    check = subprocess.run(['bcftools', 'view', '-h', str(selected_vcf)], capture_output=True, text=True)
    assert check.returncode == 0, check.stderr
    reader = VCF(str(selected_vcf))
    print('Samples:', reader.samples or '[site-only VCF]')
    print('Sequence names:', len(reader.seqnames))
    print('First contigs:', reader.seqnames[:10])
    print('Reference header:', [x for x in reader.raw_header.splitlines() if x.startswith('##reference=')][:1])
else:
    selected_vcf = None
    print('No VCF selected; skip VCF sections.')

## 4. Stream VCF into compact summary tables
This gathers aggregate counts, genotype summaries, quality/depth samples, and annotation fields when available. `MAX_METRIC_POINTS` limits plotting memory, but all variants still contribute to counts.

In [ ]:
MAX_METRIC_POINTS = 250_000
MVA_GENES = {'BUB1B', 'CEP57', 'TRIP13'}
KNOWN_MVA_RELATED = MVA_GENES | {'BUB1', 'MAD2L1', 'CDC20', 'CENPE', 'KNL1', 'NDC80', 'PLK1', 'AURKB'}

counts = Counter(); chrom_counts = Counter(); filter_counts = Counter(); consequence_counts = Counter()
sample_gt = defaultdict(Counter); qual_values, dp_values, af_values = [], [], []
candidate_rows = []

def variant_type(ref, alts):
    if len(ref) == 1 and all(len(a) == 1 for a in alts): return 'SNV'
    if any(a.startswith('<') or '[' in a or ']' in a for a in alts): return 'SV-symbolic'
    if all(len(a) != len(ref) for a in alts): return 'INDEL'
    return 'MNV/complex'

if selected_vcf:
    v = VCF(str(selected_vcf))
    samples = v.samples
    for rec in v:
        alts = [a for a in rec.ALT if a and a != '.']
        if not alts: continue
        vt = variant_type(rec.REF, alts)
        counts['variants'] += 1; counts[vt] += 1; chrom_counts[rec.CHROM] += 1
        filter_counts[rec.FILTER or 'PASS'] += 1
        if vt == 'SNV':
            pair = (rec.REF.upper(), alts[0].upper())
            if pair in {('A','G'),('G','A'),('C','T'),('T','C')}: counts['transition'] += 1
            else: counts['transversion'] += 1
        if rec.QUAL is not None and len(qual_values) < MAX_METRIC_POINTS: qual_values.append(rec.QUAL)
        dp = rec.INFO.get('DP')
        if dp is not None and len(dp_values) < MAX_METRIC_POINTS: dp_values.append(float(dp))
        af = rec.INFO.get('AF')
        if af is not None and len(af_values) < MAX_METRIC_POINTS:
            af_values.extend(np.atleast_1d(af).astype(float).tolist())
        for s, gt in zip(samples, rec.genotypes):
            alleles = gt[:-1]
            if any(a < 0 for a in alleles): label = 'missing'
            elif len(set(alleles)) == 1 and alleles[0] == 0: label = 'hom_ref'
            elif len(set(alleles)) == 1: label = 'hom_alt'
            else: label = 'het'
            sample_gt[s][label] += 1
        ann_text = ' '.join(str(rec.INFO.get(k) or '') for k in ('ANN','CSQ','BCSQ'))
        genes_hit = sorted(g for g in KNOWN_MVA_RELATED if re.search(rf'(?<![A-Z0-9]){g}(?![A-Z0-9])', ann_text.upper()))
        for term in ('stop_gained','frameshift_variant','splice_acceptor_variant','splice_donor_variant','missense_variant'):
            if term in ann_text: consequence_counts[term] += 1
        if genes_hit:
            candidate_rows.append({'CHROM':rec.CHROM,'POS':rec.POS,'REF':rec.REF,'ALT':','.join(alts),
                                   'QUAL':rec.QUAL,'FILTER':rec.FILTER or 'PASS','genes':','.join(genes_hit),
                                   'annotation_excerpt':ann_text[:500]})
    v.close()

    overview = pd.Series(counts, name='count').to_frame()
    overview.loc['Ti/Tv', 'count'] = counts['transition'] / max(counts['transversion'], 1)
    display(overview)
    gt_df = pd.DataFrame(sample_gt).T.fillna(0).astype(int)
    display(gt_df)
    pd.DataFrame(candidate_rows).to_csv(OUT/'mva_gene_hits_unfiltered.csv', index=False)
else:
    print('Skipped')

## 5. VCF visualizations
Interpret these as technical signals: chromosome distribution, variant composition, filters, quality/depth, and per-sample genotype composition. Extreme patterns can indicate calling, normalization, sex-chromosome, or contig-label issues.

In [ ]:
if selected_vcf:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    pd.Series({k:counts[k] for k in ('SNV','INDEL','MNV/complex','SV-symbolic')}).sort_values().plot.barh(ax=axes[0,0], title='Variant types')
    pd.Series(filter_counts).sort_values(ascending=False).head(12).plot.bar(ax=axes[0,1], title='FILTER values')
    chrom_order = sorted(chrom_counts, key=lambda x: (not str(x).replace('chr','').isdigit(), int(str(x).replace('chr','')) if str(x).replace('chr','').isdigit() else 999, str(x)))
    pd.Series(chrom_counts).reindex(chrom_order).plot.bar(ax=axes[1,0], title='Variants by contig')
    if qual_values: sns.histplot(qual_values, bins=60, ax=axes[1,1]); axes[1,1].set_title('QUAL distribution (sampled)')
    plt.tight_layout(); plt.savefig(OUT/'vcf_overview.png', dpi=160, bbox_inches='tight'); plt.show()

    if len(gt_df):
        gt_df.div(gt_df.sum(axis=1), axis=0).plot.bar(stacked=True, figsize=(12,5), title='Genotype proportions per sample')
        plt.ylabel('Proportion'); plt.tight_layout(); plt.savefig(OUT/'genotype_proportions.png', dpi=160); plt.show()
    if dp_values or af_values:
        fig, axes = plt.subplots(1, 2, figsize=(13,4))
        if dp_values: sns.histplot(dp_values, bins=60, ax=axes[0]); axes[0].set_xlim(0, np.percentile(dp_values, 99)); axes[0].set_title('Site depth DP (≤99th percentile)')
        if af_values: sns.histplot(af_values, bins=50, ax=axes[1]); axes[1].set_title('INFO allele frequency')
        plt.tight_layout(); plt.savefig(OUT/'vcf_depth_af.png', dpi=160); plt.show()

## 6. Annotation and MVA-oriented first pass
The gene-hit table only works if the VCF already has `ANN`, `CSQ`, or `BCSQ`. A hit is **not causal evidence**. For MVA, later analysis must examine rare damaging biallelic variants (including compound heterozygotes) in genes such as `BUB1B`, `CEP57`, and `TRIP13`, plus CNV/SV and phenotype/inheritance evidence.

In [ ]:
if selected_vcf:
    candidates = pd.DataFrame(candidate_rows)
    print('MVA-related annotated records:', len(candidates))
    display(candidates.head(50) if len(candidates) else 'No gene hits found. The VCF may be unannotated.')
    if consequence_counts:
        pd.Series(consequence_counts).sort_values().plot.barh(figsize=(8,4), title='Selected predicted consequences')
        plt.tight_layout(); plt.savefig(OUT/'consequences.png', dpi=160); plt.show()
    print('Next: annotate with VEP/SnpEff, population AF (gnomAD), ClinVar, CADD/REVEL/SpliceAI, then test inheritance using the trio.')

## 7. BAM/CRAM profiling
This checks alignment summary, insert size/error statistics, and coverage sampled every 10 kb. CRAM generally needs the exact reference FASTA; set `REFERENCE_FASTA` when required. Coverage from one tissue can suggest large CNVs/aneuploidy, but low-level mosaicism needs a dedicated method and validation.

In [ ]:
REFERENCE_FASTA = None  # e.g. Path('/kaggle/input/grch38-reference/Homo_sapiens_assembly38.fasta')
if ALIGNMENT_PATH:
    ref_args = ['--reference', str(REFERENCE_FASTA)] if REFERENCE_FASTA else []
    flagstat = subprocess.run(['samtools','flagstat',*ref_args,str(ALIGNMENT_PATH)], capture_output=True, text=True)
    print(flagstat.stdout if flagstat.returncode == 0 else flagstat.stderr)
    stats_path = OUT/'samtools_stats.txt'
    with open(stats_path, 'w') as f:
        r = subprocess.run(['samtools','stats',*ref_args,str(ALIGNMENT_PATH)], stdout=f, stderr=subprocess.PIPE, text=True)
    if r.returncode: print(r.stderr)
    # Sparse depth: one queried base per 10 kb, much cheaper than whole-base depth.
    header = subprocess.run(['samtools','view','-H',*ref_args,str(ALIGNMENT_PATH)], capture_output=True, text=True, check=True).stdout
    regions = OUT/'sparse_positions.bed'
    with open(regions, 'w') as f:
        for line in header.splitlines():
            if line.startswith('@SQ'):
                fields = dict(x.split(':',1) for x in line.split('\t')[1:])
                name, length = fields['SN'], int(fields['LN'])
                if re.fullmatch(r'(chr)?([1-9]|1[0-9]|2[0-2]|X|Y)', name):
                    for pos in range(0, length, 10_000): f.write(f'{name}\t{pos}\t{pos+1}\n')
    depth_cmd = ['samtools','depth','-a','-b',str(regions),*ref_args,str(ALIGNMENT_PATH)]
    depth = subprocess.run(depth_cmd, capture_output=True, text=True, check=True).stdout
    depth_df = pd.DataFrame([x.split('\t') for x in depth.splitlines()], columns=['chrom','pos','depth'])
    depth_df[['pos','depth']] = depth_df[['pos','depth']].astype(int)
    chrom_depth = depth_df.groupby('chrom').depth.agg(['median','mean','count']).sort_index()
    display(chrom_depth)
    chrom_depth['median'].plot.bar(figsize=(13,4), title='Sparse median depth by chromosome (10 kb grid)')
    plt.ylabel('Depth'); plt.tight_layout(); plt.savefig(OUT/'alignment_chromosome_depth.png', dpi=160); plt.show()
else:
    print('No BAM/CRAM selected; skip this section.')

## 8. FASTQ raw-read profiling
This samples the first 200,000 reads per FASTQ for read length, mean base quality, GC%, and N%. It is fast and useful for early QC, but FastQC/MultiQC should be used for a full report.

In [ ]:
def sample_fastq(path, max_reads=200_000):
    opener = gzip.open if str(path).endswith('.gz') else open
    rows = []
    with opener(path, 'rt', errors='replace') as f:
        for i in range(max_reads):
            name, seq, plus, qual = f.readline(), f.readline().strip(), f.readline(), f.readline().strip()
            if not qual: break
            rows.append((len(seq), np.mean([ord(c)-33 for c in qual]), (seq.upper().count('G')+seq.upper().count('C'))/max(len(seq),1), seq.upper().count('N')/max(len(seq),1)))
    return pd.DataFrame(rows, columns=['read_length','mean_phred','gc_fraction','n_fraction'])

fastq_summaries = []
for fq in FASTQ_PATHS:
    q = sample_fastq(fq)
    fastq_summaries.append({'file':fq.name,'sampled_reads':len(q),'mean_length':q.read_length.mean(),
                            'mean_phred':q.mean_phred.mean(),'mean_GC':q.gc_fraction.mean(),'reads_with_N_pct':100*(q.n_fraction>0).mean()})
    fig, axes = plt.subplots(1,3,figsize=(15,4))
    sns.histplot(q.read_length, bins=50, ax=axes[0]); axes[0].set_title(f'{fq.name}: read length')
    sns.histplot(q.mean_phred, bins=50, ax=axes[1]); axes[1].set_title('Mean Phred quality')
    sns.histplot(q.gc_fraction, bins=50, ax=axes[2]); axes[2].set_title('GC fraction')
    plt.tight_layout(); plt.savefig(OUT/f'{fq.name}_fastq_qc.png', dpi=160); plt.show()
display(pd.DataFrame(fastq_summaries) if fastq_summaries else 'No FASTQ selected; skip this section.')

## 9. Save compact outputs and interpretation checklist

**Questions for early insight**
1. Are call counts, Ti/Tv, PASS proportion, depth, and heterozygosity plausible for this caller and file scope?
2. Is this single-sample or trio data, and can recessive/compound-heterozygous inheritance be tested?
3. Is the VCF annotated and does it contain SNV/indel only, or also CNV/SV calls?
4. Are rare damaging variants present in `BUB1B`, `CEP57`, `TRIP13`, or related mitotic-checkpoint genes?
5. Do BAM depth and allele balance show chromosome-level or mosaic abnormalities that merit dedicated CNV/SV/mosaic analysis?
6. Do candidate variants match the child's HPO phenotype and segregation evidence?

In [ ]:
if selected_vcf:
    pd.Series(counts, name='count').to_csv(OUT/'vcf_counts.csv')
    pd.Series(chrom_counts, name='count').to_csv(OUT/'variants_by_contig.csv')
    pd.Series(filter_counts, name='count').to_csv(OUT/'filter_counts.csv')
    if len(gt_df): gt_df.to_csv(OUT/'sample_genotype_counts.csv')
print('Generated outputs:')
for p in sorted(OUT.iterdir()): print('-', p.name)
print('\nDownload files from Kaggle > Output after Save Version.')